# Phase 8 — delapan video, raw → tracking + ReID → passage → audit

Notebook prototype Kaggle ini memproses satu lokasi dengan delapan slot weekday/weekend × pagi/siang/sore/malam. **Mode awal `prepare` tidak menjalankan model.** Input aktual berada di dataset `chrisbiran/tgdi-traffic-videos`. Konvensi nama `KODETOKO-KATEGORI-KEADAAN` telah dikonfirmasi pengguna; nama toko, koordinat lokasi, dan persetujuan geometri tetap membutuhkan bukti tersendiri.

Raw dipertahankan dari keputusan Phase 7. YOLO26M, BoT-SORT, ReID dan kontrol counting B v1 dibekukan. Encoder person-domain adalah kontrol eksperimen; konfigurasi Vehicle-ReID final masih OPEN. Tidak ada tuning enhancement/deduplikasi, UI, Docker, atau orkestrasi dua GPU di notebook ini.

In [ ]:
from pathlib import Path
import json, sys, importlib, importlib.metadata, subprocess
REPO_ROOT = Path(globals().get('REPO_ROOT', Path.cwd())).resolve()
DATASET_ROOT = Path(globals().get('DATASET_ROOT', '/kaggle/input/datasets/chrisbiran/tgdi-traffic-videos'))
MANIFEST_PATH = Path(globals().get('MANIFEST_PATH', REPO_ROOT / 'docs/evidence/phase_8/config/batch_manifest_v1.json'))
PROFILE_PATH = Path(globals().get('PROFILE_PATH', REPO_ROOT / 'docs/evidence/phase_8/config/processing_profile_v1.json'))
OUTPUT_ROOT = Path(globals().get('OUTPUT_ROOT', '/kaggle/working/phase_8_batch'))
MODE = globals().get('MODE', 'prepare')  # Pilihan eksplisit: 'prepare' atau 'run_batch'.
BUILD_MANIFEST_FROM_DISCOVERY = False  # Hanya membuat usulan baru; tidak menimpa manifest.
PROPOSED_MANIFEST_PATH = Path('/kaggle/working/phase_8_proposed_manifest.json')
if MODE not in {'prepare', 'run_batch'}:
    raise ValueError('MODE harus prepare atau run_batch')
sys.path.insert(0, str(REPO_ROOT / 'notebooks'))
from phase_8_batch_helpers import prepare_batch, run_batch, configured_execution_mode, write_preparation_outputs
from phase_8_manifest_helpers import discover_videos, make_manifest_from_discovery, validate_manifest
from phase_8_pipeline_helpers import validate_processing_profile
PREPARATION_DIR = None
print('Mode:', MODE)
print('Dataset:', DATASET_ROOT)
print('Repositori:', REPO_ROOT)


## Sumber kode dan konfigurasi

Lampirkan checkout/source bundle GitHub yang memuat notebook, helper Phase 5–8, konfigurasi frozen, feedback geometri, dan manifest Phase 8. GitHub adalah sumber kode. Video dan model tidak ditambahkan ke Git. Jangan mengganti filename atau kode toko untuk mengisi fakta yang belum diketahui.

Isi delapan assignment manifest dari discovery aktual. Contoh pola yang dikonfirmasi: `TGDI-WEEKEND-MALAM` → slot `weekend_malam`, kode toko `TGDI`; ini tidak menyatakan nama toko atau koordinatnya. Per-video review harus mengikat garis/polygon, arah, SHA video dan gambar native.

In [ ]:
if not MANIFEST_PATH.is_file() or not PROFILE_PATH.is_file():
    raise FileNotFoundError('Lampirkan source bundle dan konfigurasi Phase 8: ' + str(MANIFEST_PATH.parent))
MANIFEST = json.loads(MANIFEST_PATH.read_text())
PROFILE = json.loads(PROFILE_PATH.read_text())
validate_processing_profile(PROFILE, REPO_ROOT)
PREPARATION = prepare_batch(MANIFEST_PATH, repo_root=REPO_ROOT, dataset_root=DATASET_ROOT)
print(json.dumps({k:v for k,v in PREPARATION.items() if k != 'discovered_videos'}, indent=2, ensure_ascii=False))
print('Discovery aktual (bukan inferensi weekday/weekend dari tebakan):')
print(json.dumps(PREPARATION['discovered_videos'], indent=2, ensure_ascii=False))


In [ ]:
if BUILD_MANIFEST_FROM_DISCOVERY:
    if PREPARATION['dataset_discovery_status'] != 'ACTUAL_FILES_DISCOVERED_METADATA_UNCONFIRMED':
        raise RuntimeError('Usulan manifest memerlukan discovery aktual yang tersedia')
    proposed = make_manifest_from_discovery(PREPARATION['discovered_videos'], dataset_root=DATASET_ROOT)
    if PROPOSED_MANIFEST_PATH.exists():
        raise FileExistsError('Tidak menimpa konfigurasi yang sudah ada: ' + str(PROPOSED_MANIFEST_PATH))
    PROPOSED_MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)
    PROPOSED_MANIFEST_PATH.write_text(json.dumps(proposed, indent=2, ensure_ascii=False) + '\n')
    print('Usulan tersimpan:', PROPOSED_MANIFEST_PATH)
    print('Lengkapi metadata lokasi dan bukti review geometri; usulan ini belum mengizinkan inference.')
else:
    print('Manifest tersimpan tetap digunakan; tidak ada assignment atau approval baru yang dibuat.')


## Gate sebelum eksekusi

`prepare` menampilkan isu tanpa mengesahkan file. `run_batch` memeriksa delapan sumber, SHA, seluruh decode, ukuran native, metadata lokasi, slot unik, dan review geometri yang terikat video. Template G3/N1 dapat menjadi usulan awal; persetujuan TOR3 bukan persetujuan TGDI.

Frame count, FPS, dan posisi garis berasal dari inspeksi nyata. Gap/prediksi/endpoint ambigu, konflik kelas dan identitas tetap pending. Full-video counting baru tidak menggunakan kembali overlay tujuh passage Phase 7 sebagai hasil seluruh video.

Untuk penelitian yang secara eksplisit diizinkan, nama toko/latitude/longitude dapat tetap null apabila manifest memuat acknowledgement pengguna yang terikat delapan sumber. Mode ini tetap berlabel penelitian dengan metadata bisnis belum lengkap; nilai yang belum diketahui tidak diisi dengan tebakan.

In [ ]:
READINESS = validate_manifest(MANIFEST, repo_root=REPO_ROOT, check_files=(MODE == 'run_batch'))
print(json.dumps(READINESS, indent=2, ensure_ascii=False))
if MODE == 'run_batch' and not READINESS['ready']:
    raise RuntimeError('Batch belum boleh dijalankan: lengkapi isu manifest di atas terlebih dahulu.')


## Pengujian software

Fixture sintetis memeriksa kontrak input, geometri/crossing, source provenance, duplicate passage, pemisahan snapshot machine/reviewed, pending, failure/null, dan rekonsiliasi ledger→CSV. Hasil tes software tidak mengukur kinerja model atau kendaraan fisik.

In [ ]:
import unittest, io
GUARDS = {}
for module_name in ('phase_8_manifest_tests', 'phase_8_pipeline_tests', 'phase_8_batch_tests', 'phase_8_review_tests'):
    module = importlib.import_module(module_name)
    suite = unittest.defaultTestLoader.loadTestsFromModule(module)
    log = io.StringIO()
    result = unittest.TextTestRunner(stream=log, verbosity=2).run(suite)
    report = {'evidence_kind':'SYNTHETIC_TEST_FIXTURE', 'tests_run':result.testsRun,
              'tests_passed':result.testsRun - len(result.failures) - len(result.errors) - len(result.skipped),
              'tests_skipped':len(result.skipped),
              'successful':result.wasSuccessful() and result.testsRun - len(result.skipped) > 0, 'log':log.getvalue()}
    GUARDS[module_name] = report
    if not report['successful']:
        raise RuntimeError('Software guard gagal: ' + module_name + '\n' + json.dumps(report, indent=2))
    print(module_name, report['tests_passed'], '/', report['tests_run'], 'software tests; skipped:', report['tests_skipped'])
if MODE == 'run_batch':
    guard_dir = OUTPUT_ROOT / 'pre_execution_guard'
    guard_dir.mkdir(parents=True, exist_ok=True)
    (guard_dir / 'synthetic_validation.json').write_text(json.dumps(GUARDS, indent=2, ensure_ascii=False) + '\n')


In [ ]:
if MODE == 'prepare':
    PREPARATION_REPORT = write_preparation_outputs(
        PREPARATION, GUARDS, MANIFEST_PATH, PROFILE_PATH, OUTPUT_ROOT,
        repo_root=REPO_ROOT, notebook_path='notebooks/08_end_to_end_batch.ipynb')
    PREPARATION_DIR = Path(PREPARATION_REPORT['output_directory'])
    print('Evidence preparation tersimpan:', PREPARATION_DIR)
    print('Tidak ada inference model, tabel traffic, atau klaim delapan video selesai.')


## Device dan dependency

GPU tersedia ditemukan secara dinamis, tetapi assignment eksperimen ini tetap **satu video aktif, CUDA0** untuk inference baru. Dua GPU terlihat tidak berarti keduanya digunakan. Semua sumber dengan cache eksplisit yang telah diverifikasi dapat direplay tanpa GPU lokal. Cache yang tidak valid menghasilkan kegagalan; tidak ada fallback diam-diam ke inference.

Jika lingkungan sudah mengimpor versi dependency lain, restart session terlebih dahulu. Model resmi diunduh dan diverifikasi SHA oleh pipeline. Raw frame yang sama digunakan detector dan encoder.

In [ ]:
EXECUTION_MODES = {v['video_id']: configured_execution_mode(v) for v in MANIFEST['videos']}
NEEDS_NEW_INFERENCE = MODE == 'run_batch' and any(v == 'new_inference' for v in EXECUTION_MODES.values())
print('Rute eksplisit:', EXECUTION_MODES)
if NEEDS_NEW_INFERENCE:
    wanted = {'ultralytics':'8.4.175', 'lap':'0.5.13', 'onnx':'1.20.1', 'onnxruntime-gpu':'1.24.4'}
    missing = []
    for package, version in wanted.items():
        try: installed = importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError: installed = None
        if installed != version:
            module = 'onnxruntime' if package == 'onnxruntime-gpu' else package.replace('-', '_')
            if module in sys.modules:
                raise RuntimeError(package + ' sudah diimpor dengan versi berbeda; restart session')
            missing.append(package + '==' + version)
    if missing:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', *missing], check=True)
    from phase_7_night_helpers import verify_gpu_device
    print(verify_gpu_device(0))
else:
    print('Tidak ada loading model/GPU baru pada sel ini.')


## Eksekusi berurutan

Untuk source baru, gunakan `execution_mode="new_inference"` (default eksplisit notebook). Untuk cache yang sudah mempunyai bukti model/tracker/ReID yang sesuai, gunakan `execution_mode="verified_cache_replay"` dan `retained_cache={records_path, cache_identity, cache_provenance}`. `cache_provenance` mengikat report GPU aktual, SHA, appearance records dan namespace measurement. Cache detection saja bukan pengganti cache tracking + ReID.

Setiap sumber diproses sampai EOF. Output mencakup event accepted, keputusan pending/rejected, annotated video dan machine-readable logs. Kegagalan satu video dicatat dengan subtotal null; hasil parsial tidak disebut batch lengkap.

In [ ]:
BATCH_REPORT = None
if MODE == 'run_batch':
    BATCH_REPORT = run_batch(MANIFEST_PATH, PROFILE_PATH, OUTPUT_ROOT, repo_root=REPO_ROOT)
    print('Run:', BATCH_REPORT['run_id'])
    print('Status:', BATCH_REPORT['status'])
    print('Output:', BATCH_REPORT['output_directory'])
    print('Accepted observed:', BATCH_REPORT['accepted_observed_event_count'])
    print('Pending observed:', BATCH_REPORT['pending_observed_count'])
else:
    print('PREPARE ONLY: tidak ada video yang diproses model dan tidak ada hasil traffic yang dibuat.')


## Tabel hasil dan batas interpretasi

`weekday.csv` dan `weekend.csv` berisi `category,pagi,siang,sore,malam,total`. Angka berasal hanya dari satu snapshot accepted yang dipilih per video. Default snapshot adalah machine; setelah audit, pilih snapshot reviewed secara eksplisit. Machine dan reviewed tidak dijumlahkan.

Kategori pedestrian belum ditangani resolver kendaraan B v1; taxonomy bus/angkot masih OPEN. Sel tersebut null/kosong, bukan 0. Sel didukung bernilai 0 berarti belum ada accepted event pada snapshot itu, bukan bukti tidak ada kendaraan. Pending berada di audit queue di luar subtotal. Total menjumlah empat rekaman yang diamati, bukan estimasi lalu lintas sehari atau hasil accuracy.

In [ ]:
if BATCH_REPORT is not None:
    print(json.dumps({'tables': BATCH_REPORT['tables'],
                      'coverage': BATCH_REPORT['coverage_by_category'],
                      'complete_batch_accepted_total': BATCH_REPORT['complete_batch_accepted_total'],
                      'all_eight_videos_processed': BATCH_REPORT['all_eight_videos_processed']}, indent=2, ensure_ascii=False))
    output = Path(BATCH_REPORT['output_directory'])
    for name in ('weekday.csv', 'weekend.csv', 'accepted_event_ledger.json', 'audit_queue.json',
                 'coverage.json', 'per_video_results.json', 'batch_retention_manifest.json'):
        path = output / name
        print(name, '=>', path, '(tersedia)' if path.is_file() else '(tidak tersedia)')


## Review berikutnya

Periksa annotated video dan kasus pending per passage: physical unit, rider+motorcycle, duplikasi, kategori, sumber BC, geometri, dan crossing. `phase_8_review_helpers.make_review_template()` membuat antrean kosong; isi review hanya setelah melihat bukti native aktual. `apply_review()` menjaga snapshot machine dan menghasilkan snapshot reviewed terpisah. Review tidak memperbaiki crossing gap/endpoint/prediksi yang tidak valid.

Phase 8 selesai secara eksekusi hanya setelah delapan input yang disepakati benar-benar diproses, output video/log tersedia, dan agregat direkonsiliasi. Audit lebih luas belongs Phase 9. Notebook prepare, tes sintetis, atau pilot satu TOR3 bukan bukti bahwa seluruh delapan TGDI sudah selesai.